# 🏆 BTK Datathon 2026 — Career Success Score (v3)

**Metrik:** RMSE  
**Donanım:** NVIDIA CUDA GPU

## v3'te Neler Yeni?

v2'ye ek olarak:

1. **🎯 Rol-Yetenek Mühendisliği** (+0.12 RMSE doğrulanmış kazanım)
   - `matched_skill` — hedef role uygun teknik skor
   - `role_composite` — rol için ağırlıklı yetenek profili
   - `role_composite_minus_avg` — fazla yetenek farkı (en güçlü tek özellik!)
   - Rol-bazlı interaksiyon özellikleri

2. **🔬 ASHA Pruner (SuccessiveHalvingPruner)** — Optuna'da daha akıllı arama

3. **🤖 5 Base Model** — ExtraTrees eklendi (varyans için)

4. **🔄 Pseudo-Labeling** — Test setini eğitime kat (confident tahminleri)

5. **📈 Genişletilmiş NLP** — 60 word + 40 char SVD component

6. **🎲 Multi-seed averaging** — 3 seed × her model

**Beklenen iyileşme:** ~-0.3 to -0.5 RMSE (89.54 → 90.5+)

In [ ]:
# ── Kütüphaneler ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings, gc, time
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from scipy.optimize import minimize

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import optuna
from optuna.pruners import SuccessiveHalvingPruner, MedianPruner
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED       = 42
N_FOLDS    = 5
N_SEEDS    = 3
TARGET_COL = 'career_success_score'

np.random.seed(SEED)
print('Kütüphaneler hazır ✓')

In [ ]:
# ── GPU Kontrol ──────────────────────────────────────────────────────────────
GPU_AVAILABLE = False
LGB_GPU = False
CAT_GPU = False

test_X = np.random.rand(100, 5).astype(np.float32)
test_y = np.random.rand(100).astype(np.float32)

try:
    m = xgb.XGBRegressor(n_estimators=10, device='cuda', tree_method='hist')
    m.fit(test_X, test_y); _ = m.predict(test_X)
    GPU_AVAILABLE = True
    print('✓ XGBoost GPU çalışıyor')
except Exception as e:
    print(f'✗ XGBoost CPU: {str(e)[:60]}')

try:
    m = lgb.LGBMRegressor(n_estimators=10, device='gpu', verbose=-1)
    m.fit(test_X, test_y)
    LGB_GPU = True
    print('✓ LightGBM GPU çalışıyor')
except Exception:
    print('✗ LightGBM CPU (GPU build yok)')

try:
    m = CatBoostRegressor(iterations=10, task_type='GPU', devices='0', verbose=0)
    m.fit(test_X, test_y)
    CAT_GPU = True
    print('✓ CatBoost GPU çalışıyor')
except Exception:
    print('✗ CatBoost CPU')

def xgb_gpu_params():
    return {'device': 'cuda', 'tree_method': 'hist'} if GPU_AVAILABLE else {'tree_method': 'hist', 'n_jobs': -1}
def lgb_gpu_params():
    return {'device': 'gpu'} if LGB_GPU else {'n_jobs': -1}
def cat_gpu_params():
    return {'task_type': 'GPU', 'devices': '0'} if CAT_GPU else {}

## 1. Veri Yükleme

In [ ]:
TRAIN_PATH  = 'train.csv'
TEST_PATH   = 'test_x.csv'
SAMPLE_PATH = 'sample_submission.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_PATH)

print(f'Train: {train.shape}, Test: {test.shape}')
y          = train[TARGET_COL].values
test_ids   = test['student_id'].values

# Hızlı stratified split için bin'ler
y_bin = pd.qcut(y, q=10, labels=False, duplicates='drop')
skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS = list(skf.split(train, y_bin))
print(f'Target: μ={y.mean():.2f}, σ={y.std():.2f}, [{y.min():.0f}, {y.max():.0f}]')

## 2. 🎯 Rol-Yetenek Mühendisliği (v3 yeni - en güçlü kazanım)

Her rol için **ağırlıklı yetenek profili** tanımlanır. Bir öğrencinin:
- `matched_skill`: hedef role uyan teknik skoru (Cloud Engineer ise cloud_score)
- `role_composite`: rolün ihtiyaç duyduğu yetenek karışımı (ağırlıklı ortalama)
- `role_composite_minus_avg`: rolün ihtiyacına göre öğrencinin fazlalığı/eksikliği

Ağaç modelleri bunu tek başlarına öğrenemiyor — domain bilgisi olarak veriyoruz.

In [ ]:
# Her rol için: {yetenek: ağırlık}
ROLE_PROFILES = {
    'Cloud Engineer':        {'cloud_score': 3, 'devops_score': 2, 'problem_solving_score': 1},
    'DevOps Engineer':       {'devops_score': 3, 'cloud_score': 2, 'problem_solving_score': 1},
    'MLOps Engineer':        {'devops_score': 2, 'machine_learning_score': 2, 'cloud_score': 1, 'problem_solving_score': 1},
    'Frontend Developer':    {'frontend_score': 3, 'coding_score': 2, 'problem_solving_score': 1},
    'Backend Developer':     {'backend_score': 3, 'coding_score': 2, 'sql_score': 1, 'problem_solving_score': 1},
    'Software Developer':    {'coding_score': 2, 'problem_solving_score': 2, 'data_structures_score': 2, 'backend_score': 1, 'frontend_score': 1},
    'Data Scientist':        {'machine_learning_score': 3, 'sql_score': 2, 'problem_solving_score': 2, 'data_structures_score': 1},
    'AI Engineer':           {'machine_learning_score': 3, 'coding_score': 2, 'problem_solving_score': 1, 'data_structures_score': 1},
    'Data Analyst':          {'sql_score': 3, 'data_structures_score': 1, 'problem_solving_score': 1},
    'Product Analyst':       {'sql_score': 2, 'problem_solving_score': 2},
    'Cybersecurity Analyst': {'devops_score': 2, 'problem_solving_score': 2, 'coding_score': 1},
}

# Tek-yetenek mapping (en önemli yetenek)
PRIMARY_SKILL_MAP = {
    'Cloud Engineer': 'cloud_score',         'DevOps Engineer': 'devops_score',
    'MLOps Engineer': 'devops_score',        'Frontend Developer': 'frontend_score',
    'Backend Developer': 'backend_score',    'Software Developer': 'coding_score',
    'Data Scientist': 'machine_learning_score', 'AI Engineer': 'machine_learning_score',
    'Data Analyst': 'sql_score',             'Product Analyst': 'sql_score',
    'Cybersecurity Analyst': 'devops_score',
}

def add_role_features(df):
    df = df.copy()
    
    # 1. matched_skill — rolün birincil yeteneği
    df['matched_skill'] = 0.0
    for role, skill in PRIMARY_SKILL_MAP.items():
        mask = df['target_role'] == role
        df.loc[mask, 'matched_skill'] = df.loc[mask, skill].values
    
    # 2. role_composite — rolün ağırlıklı yetenek karışımı
    df['role_composite'] = 0.0
    for role, weights in ROLE_PROFILES.items():
        mask = df['target_role'] == role
        if mask.sum() == 0:
            continue
        total = sum(weights.values())
        composite = sum(df.loc[mask, skill] * w for skill, w in weights.items()) / total
        df.loc[mask, 'role_composite'] = composite.values
    
    return df

train = add_role_features(train)
test  = add_role_features(test)
print('Role-skill features eklendi ✓')
print(f'matched_skill corr with y: {np.corrcoef(train["matched_skill"], y)[0,1]:.4f}')
print(f'role_composite corr with y: {np.corrcoef(train["role_composite"], y)[0,1]:.4f}')

## 3. NLP — Genişletilmiş TF-IDF

In [ ]:
def build_nlp_features(train_df, test_df, word_n=60, char_n=40):
    all_texts = pd.concat([
        train_df['mentor_feedback_text'],
        test_df['mentor_feedback_text']
    ]).fillna('').reset_index(drop=True)

    # Word-level (uni + bigram)
    word_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=3,
                                  sublinear_tf=True, analyzer='word')
    word_m = word_tfidf.fit_transform(all_texts)
    word_svd = TruncatedSVD(n_components=word_n, random_state=SEED)
    word_feats = word_svd.fit_transform(word_m)
    print(f'  Word TF-IDF → SVD: {word_svd.explained_variance_ratio_.sum():.3f} explained')

    # Char-level (Türkçe morfoloji)
    char_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(3, 5), min_df=3,
                                  sublinear_tf=True, analyzer='char_wb')
    char_m = char_tfidf.fit_transform(all_texts)
    char_svd = TruncatedSVD(n_components=char_n, random_state=SEED)
    char_feats = char_svd.fit_transform(char_m)
    print(f'  Char TF-IDF → SVD: {char_svd.explained_variance_ratio_.sum():.3f} explained')

    word_cols = [f'nlp_w_{i}' for i in range(word_n)]
    char_cols = [f'nlp_c_{i}' for i in range(char_n)]
    nlp_df = pd.DataFrame(np.hstack([word_feats, char_feats]), columns=word_cols + char_cols)
    nlp_df['text_len']   = all_texts.str.len().values
    nlp_df['word_count'] = all_texts.str.split().str.len().values

    return (nlp_df.iloc[:len(train_df)].reset_index(drop=True),
            nlp_df.iloc[len(train_df):].reset_index(drop=True))

t0 = time.time()
train_nlp, test_nlp = build_nlp_features(train, test)
print(f'NLP süresi: {time.time()-t0:.1f}s | shape: {train_nlp.shape}')

## 4. Target Encoding (sızıntısız)

In [ ]:
def kfold_target_encode(train_df, test_df, col, target, splits, smoothing=20):
    global_mean = train_df[target].mean()
    oof_enc = np.zeros(len(train_df))
    for tr_idx, val_idx in splits:
        tr = train_df.iloc[tr_idx]
        agg = tr.groupby(col)[target].agg(['mean', 'count'])
        agg['s'] = (agg['mean']*agg['count'] + global_mean*smoothing) / (agg['count']+smoothing)
        val = train_df.iloc[val_idx]
        oof_enc[val_idx] = val[col].map(agg['s']).fillna(global_mean).values
    agg = train_df.groupby(col)[target].agg(['mean', 'count'])
    agg['s'] = (agg['mean']*agg['count'] + global_mean*smoothing) / (agg['count']+smoothing)
    test_enc = test_df[col].map(agg['s']).fillna(global_mean).values
    return oof_enc, test_enc

te_cols = ['department', 'target_role', 'hobby', 'preferred_social_media_platform', 'university_tier']
te_train, te_test = {}, {}
train_with_y = train.assign(**{TARGET_COL: y})
for col in te_cols:
    tr_enc, te_enc = kfold_target_encode(train_with_y, test, col, TARGET_COL, FOLD_SPLITS, smoothing=20)
    te_train[f'te_{col}'] = tr_enc
    te_test[f'te_{col}']  = te_enc
    print(f'  te_{col}: corr={np.corrcoef(tr_enc, y)[0,1]:.3f}')
train_te = pd.DataFrame(te_train); test_te = pd.DataFrame(te_test)

## 5. Feature Engineering — Tüm Özellikler

In [ ]:
TIER_MAP  = {'Tier 1': 4, 'Tier 2': 3, 'Tier 3': 2, 'Tier 4': 1}
TECH_COLS = ['coding_score','problem_solving_score','data_structures_score',
             'sql_score','machine_learning_score','backend_score','frontend_score',
             'cloud_score','devops_score']
SOFT_COLS = ['communication_score','teamwork_score','leadership_score','presentation_score']
CAT_COLS  = ['department','target_role','hobby','preferred_social_media_platform']

def build_features(train_df, test_df):
    train_df = train_df.copy()
    test_df  = test_df.copy()

    # Null doldurma — sadece train medyanı
    fill_cols = ['english_exam_score','internship_duration_months','portfolio_score',
                 'github_avg_stars','open_source_contribution_count',
                 'linkedin_profile_score','hr_interview_score']
    medians = {c: train_df[c].median() for c in fill_cols}
    for c in fill_cols:
        train_df[c] = train_df[c].fillna(medians[c])
        test_df[c]  = test_df[c].fillna(medians[c])

    def derive(df):
        df = df.copy()
        # Teknik aggregateler
        df['tech_mean']  = df[TECH_COLS].mean(axis=1)
        df['tech_max']   = df[TECH_COLS].max(axis=1)
        df['tech_min']   = df[TECH_COLS].min(axis=1)
        df['tech_std']   = df[TECH_COLS].std(axis=1)
        df['tech_sum']   = df[TECH_COLS].sum(axis=1)
        df['tech_range'] = df['tech_max'] - df['tech_min']
        df['tech_top3']  = df[TECH_COLS].apply(lambda r: r.nlargest(3).mean(), axis=1)
        df['tech_bot3']  = df[TECH_COLS].apply(lambda r: r.nsmallest(3).mean(), axis=1)

        # Soft aggregateler
        df['soft_mean'] = df[SOFT_COLS].mean(axis=1)
        df['soft_sum']  = df[SOFT_COLS].sum(axis=1)
        df['soft_std']  = df[SOFT_COLS].std(axis=1)
        df['soft_min']  = df[SOFT_COLS].min(axis=1)

        # 🎯 Rol-yetenek interaksiyonları (v3 yeni — en güçlü)
        df['matched_x_proj']        = df['matched_skill'] * df['project_quality_score']
        df['matched_x_interview']   = df['matched_skill'] * df['technical_interview_score']
        df['role_comp_x_proj']      = df['role_composite'] * df['project_quality_score']
        df['role_comp_x_interview'] = df['role_composite'] * df['technical_interview_score']
        df['matched_minus_tech']    = df['matched_skill'] - df['tech_mean']  # rol için fazla yetenek
        df['role_comp_minus_avg']   = df['role_composite'] - df['tech_mean'] # ★ EN GÜÇLÜ (+0.11)
        df['matched_x_role_comp']   = df['matched_skill'] * df['role_composite'] / 100

        # Mülakat
        df['interview_composite'] = df['technical_interview_score']*0.6 + df['hr_interview_score']*0.4
        df['interview_diff']      = df['technical_interview_score'] - df['hr_interview_score']

        # Deneyim
        df['total_projects']  = df['real_client_project_count'] + df['freelance_project_count']
        df['experience_score']= (df['internship_count']*3 + df['real_client_project_count']*2 + 
                                  df['freelance_project_count'])
        df['exp_per_year']    = df['experience_score'] / (df['age'] - 18).clip(lower=1)

        # GitHub / OSS
        df['github_score']    = (df['github_repo_count'] * np.log1p(df['github_avg_stars']+1) +
                                  df['open_source_contribution_count'])
        df['oss_ratio']       = df['open_source_contribution_count'] / (df['github_repo_count']+1)

        # Profil
        df['profile_composite']= (df['portfolio_score'] + df['linkedin_profile_score'] + df['cv_quality_score'])/3
        df['profile_min']      = df[['portfolio_score','linkedin_profile_score','cv_quality_score']].min(axis=1)
        df['profile_max']      = df[['portfolio_score','linkedin_profile_score','cv_quality_score']].max(axis=1)

        # Hackathon ve başvuru
        df['hackathon_eff']   = np.where(df['hackathon_count']>0, 
                                          df['hackathon_awards']/df['hackathon_count'], 0)
        df['app_success_rate']= df['interviews_attended'] / (df['applications_sent']+1)

        # Eğitim / üni
        df['university_tier_num'] = df['university_tier'].map(TIER_MAP)
        df['years_to_grad'] = df['graduation_year'] - df['application_year']
        df['cert_per_internship'] = df['certification_count'] / (df['internship_count']+1)
        df['cgpa_x_attendance']   = df['cgpa'] * df['attendance_rate']
        df['failed_penalty']      = df['failed_courses_count'] / (df['cgpa']+0.01)

        # Genel interaksiyonlar
        df['soft_x_technical']    = df['soft_mean'] * df['technical_interview_score']
        df['proj_x_interview']    = df['project_quality_score'] * df['technical_interview_score']
        df['proj_x_portfolio']    = df['project_quality_score'] * df['portfolio_score']

        # Domain composite
        df['overall_score'] = (df['tech_mean']*0.25 + df['soft_mean']*0.15 +
                                df['project_quality_score']*0.25 +
                                df['profile_composite']*0.10 +
                                df['interview_composite']*0.15 +
                                df['matched_skill']*0.10)
        return df

    train_df = derive(train_df)
    test_df  = derive(test_df)

    # CatBoost native categorical için kopyala
    train_cat = train_df.copy(); test_cat = test_df.copy()

    # Label encoding (combined fit)
    for col in CAT_COLS + ['university_tier']:
        le = LabelEncoder()
        le.fit(pd.concat([train_df[col], test_df[col]]).astype(str))
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col]  = le.transform(test_df[col].astype(str))

    drop_cols = ['student_id', 'mentor_feedback_text']
    if TARGET_COL in train_df.columns: drop_cols.append(TARGET_COL)
    train_df = train_df.drop(columns=drop_cols, errors='ignore')
    test_df  = test_df.drop(columns=drop_cols, errors='ignore')
    train_cat= train_cat.drop(columns=drop_cols, errors='ignore')
    test_cat = test_cat.drop(columns=drop_cols, errors='ignore')

    return train_df, test_df, train_cat, test_cat

t0 = time.time()
X_train, X_test, X_train_cat, X_test_cat = build_features(train, test)

# NLP + TE ekle
X_train = pd.concat([X_train.reset_index(drop=True), train_nlp, train_te], axis=1)
X_test  = pd.concat([X_test.reset_index(drop=True),  test_nlp,  test_te],  axis=1)
X_train_cat = pd.concat([X_train_cat.reset_index(drop=True), train_nlp, train_te], axis=1)
X_test_cat  = pd.concat([X_test_cat.reset_index(drop=True),  test_nlp,  test_te],  axis=1)

# Null doldurma (sadece sayısal)
col_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(col_medians)
X_test  = X_test.fillna(col_medians)
for c in X_train_cat.columns:
    if pd.api.types.is_numeric_dtype(X_train_cat[c]) and X_train_cat[c].isna().any():
        X_train_cat[c] = X_train_cat[c].fillna(col_medians.get(c, 0))
        X_test_cat[c]  = X_test_cat[c].fillna(col_medians.get(c, 0))

print(f'FE süresi: {time.time()-t0:.1f}s')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
cat_features_idx = [X_train_cat.columns.get_loc(c) for c in CAT_COLS + ['university_tier']]

## 6. Optuna — SuccessiveHalvingPruner (ASHA)

v2'deki MedianPruner yerine **SuccessiveHalvingPruner** kullanıyoruz — daha agresif kesim, daha çok trial verimli denenir.

In [ ]:
# Hızlı 3-fold (sadece Optuna için)
opt_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
OPT_SPLITS = list(opt_skf.split(X_train, y_bin))

def make_pruner():
    return SuccessiveHalvingPruner(min_resource=50, reduction_factor=4)

# ── XGBoost ───────────────────────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators'     : 3000,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 4, 10),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 20),
        'gamma'            : trial.suggest_float('gamma', 0, 2.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'early_stopping_rounds': 50, 'random_state': SEED, **xgb_gpu_params()
    }
    rmses = []
    for fold_id, (tr_idx, val_idx) in enumerate(OPT_SPLITS):
        m = xgb.XGBRegressor(**params)
        m.fit(X_train.iloc[tr_idx], y[tr_idx],
              eval_set=[(X_train.iloc[val_idx], y[val_idx])], verbose=False)
        rmses.append(np.sqrt(mean_squared_error(y[val_idx], m.predict(X_train.iloc[val_idx]))))
        trial.report(np.mean(rmses), fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(rmses)

print('XGBoost Optuna (ASHA, 60 trial)...')
xgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED),
                                  pruner=make_pruner())
t0 = time.time()
xgb_study.optimize(xgb_objective, n_trials=60, show_progress_bar=True)
print(f'XGB: {xgb_study.best_value:.4f} RMSE  ({time.time()-t0:.0f}s, {len(xgb_study.trials)} trials)')

In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────────
def lgb_objective(trial):
    params = {
        'n_estimators': 3000,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 31, 400),
        'max_depth'        : trial.suggest_int('max_depth', 4, 12),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': SEED, 'verbose': -1, **lgb_gpu_params()
    }
    rmses = []
    for fold_id, (tr_idx, val_idx) in enumerate(OPT_SPLITS):
        m = lgb.LGBMRegressor(**params)
        m.fit(X_train.iloc[tr_idx], y[tr_idx],
              eval_set=[(X_train.iloc[val_idx], y[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        rmses.append(np.sqrt(mean_squared_error(y[val_idx], m.predict(X_train.iloc[val_idx]))))
        trial.report(np.mean(rmses), fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return np.mean(rmses)

print('LightGBM Optuna (ASHA, 60 trial)...')
lgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), pruner=make_pruner())
t0 = time.time()
lgb_study.optimize(lgb_objective, n_trials=60, show_progress_bar=True)
print(f'LGB: {lgb_study.best_value:.4f} RMSE  ({time.time()-t0:.0f}s)')

In [ ]:
# ── CatBoost ──────────────────────────────────────────────────────────────────
def cat_objective(trial):
    params = {
        'iterations'       : 3000,
        'learning_rate'    : trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth'            : trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg'      : trial.suggest_float('l2_leaf_reg', 1, 15),
        'min_data_in_leaf' : trial.suggest_int('min_data_in_leaf', 1, 100),
        'random_strength'  : trial.suggest_float('random_strength', 0.5, 5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_seed': SEED, 'verbose': 0, 'loss_function': 'RMSE',
        'early_stopping_rounds': 50, **cat_gpu_params()
    }
    rmses = []
    for fold_id, (tr_idx, val_idx) in enumerate(OPT_SPLITS):
        m = CatBoostRegressor(**params)
        m.fit(X_train_cat.iloc[tr_idx], y[tr_idx],
              eval_set=(X_train_cat.iloc[val_idx], y[val_idx]),
              cat_features=cat_features_idx, verbose=0)
        rmses.append(np.sqrt(mean_squared_error(y[val_idx], m.predict(X_train_cat.iloc[val_idx]))))
        trial.report(np.mean(rmses), fold_id)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(rmses)

print('CatBoost Optuna (ASHA, 40 trial)...')
cat_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED), pruner=make_pruner())
t0 = time.time()
cat_study.optimize(cat_objective, n_trials=40, show_progress_bar=True)
print(f'CAT: {cat_study.best_value:.4f} RMSE  ({time.time()-t0:.0f}s)')

In [ ]:
# ── HistGradientBoosting ──────────────────────────────────────────────────────
def hgb_objective(trial):
    params = {
        'max_iter'         : trial.suggest_int('max_iter', 500, 2000),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 4, 12),
        'max_leaf_nodes'   : trial.suggest_int('max_leaf_nodes', 15, 300),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 5, 100),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-3, 10, log=True),
        'random_state': SEED
    }
    rmses = []
    for tr_idx, val_idx in OPT_SPLITS:
        m = HistGradientBoostingRegressor(**params)
        m.fit(X_train.iloc[tr_idx], y[tr_idx])
        rmses.append(np.sqrt(mean_squared_error(y[val_idx], m.predict(X_train.iloc[val_idx]))))
    return np.mean(rmses)

print('HistGB Optuna (30 trial)...')
hgb_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED),
                                  pruner=MedianPruner(n_startup_trials=5))
t0 = time.time()
hgb_study.optimize(hgb_objective, n_trials=30, show_progress_bar=True)
print(f'HGB: {hgb_study.best_value:.4f} RMSE  ({time.time()-t0:.0f}s)')

In [ ]:
# ── ExtraTrees (yeni, sade) ────────────────────────────────────────────────────
def et_objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 300, 1000),
        'max_depth'       : trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features'    : trial.suggest_float('max_features', 0.3, 1.0),
        'random_state': SEED, 'n_jobs': -1
    }
    rmses = []
    for tr_idx, val_idx in OPT_SPLITS:
        m = ExtraTreesRegressor(**params)
        m.fit(X_train.iloc[tr_idx], y[tr_idx])
        rmses.append(np.sqrt(mean_squared_error(y[val_idx], m.predict(X_train.iloc[val_idx]))))
    return np.mean(rmses)

print('ExtraTrees Optuna (20 trial)...')
et_study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
t0 = time.time()
et_study.optimize(et_objective, n_trials=20, show_progress_bar=True)
print(f'ET: {et_study.best_value:.4f} RMSE  ({time.time()-t0:.0f}s)')

## 7. Stacking — 5 Model × Multi-Seed

In [ ]:
def stack_xgb(params, splits, X_tr, X_te, y_, n_seeds=N_SEEDS):
    oof, test_p = np.zeros(len(y_)), np.zeros(len(X_te))
    for seed in range(n_seeds):
        p = {**params, 'random_state': SEED+seed, **xgb_gpu_params(),
             'early_stopping_rounds': 50, 'n_estimators': 3000}
        fold_te = np.zeros(len(X_te))
        for tr_idx, val_idx in splits:
            m = xgb.XGBRegressor(**p)
            m.fit(X_tr.iloc[tr_idx], y_[tr_idx],
                  eval_set=[(X_tr.iloc[val_idx], y_[val_idx])], verbose=False)
            oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
            fold_te += m.predict(X_te) / len(splits)
        test_p += fold_te / n_seeds
    return oof, test_p

def stack_lgb(params, splits, X_tr, X_te, y_, n_seeds=N_SEEDS):
    oof, test_p = np.zeros(len(y_)), np.zeros(len(X_te))
    for seed in range(n_seeds):
        p = {**params, 'random_state': SEED+seed, **lgb_gpu_params(),
             'n_estimators': 3000, 'verbose': -1}
        fold_te = np.zeros(len(X_te))
        for tr_idx, val_idx in splits:
            m = lgb.LGBMRegressor(**p)
            m.fit(X_tr.iloc[tr_idx], y_[tr_idx],
                  eval_set=[(X_tr.iloc[val_idx], y_[val_idx])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
            fold_te += m.predict(X_te) / len(splits)
        test_p += fold_te / n_seeds
    return oof, test_p

def stack_cat(params, splits, X_tr, X_te, y_, cat_idx, n_seeds=N_SEEDS):
    oof, test_p = np.zeros(len(y_)), np.zeros(len(X_te))
    for seed in range(n_seeds):
        p = {**params, 'random_seed': SEED+seed, **cat_gpu_params(),
             'iterations': 3000, 'verbose': 0, 'loss_function': 'RMSE',
             'early_stopping_rounds': 50}
        fold_te = np.zeros(len(X_te))
        for tr_idx, val_idx in splits:
            m = CatBoostRegressor(**p)
            m.fit(X_tr.iloc[tr_idx], y_[tr_idx],
                  eval_set=(X_tr.iloc[val_idx], y_[val_idx]),
                  cat_features=cat_idx, verbose=0)
            oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
            fold_te += m.predict(X_te) / len(splits)
        test_p += fold_te / n_seeds
    return oof, test_p

def stack_hgb(params, splits, X_tr, X_te, y_, n_seeds=N_SEEDS):
    oof, test_p = np.zeros(len(y_)), np.zeros(len(X_te))
    for seed in range(n_seeds):
        p = {**params, 'random_state': SEED+seed}
        fold_te = np.zeros(len(X_te))
        for tr_idx, val_idx in splits:
            m = HistGradientBoostingRegressor(**p)
            m.fit(X_tr.iloc[tr_idx], y_[tr_idx])
            oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
            fold_te += m.predict(X_te) / len(splits)
        test_p += fold_te / n_seeds
    return oof, test_p

def stack_et(params, splits, X_tr, X_te, y_, n_seeds=N_SEEDS):
    oof, test_p = np.zeros(len(y_)), np.zeros(len(X_te))
    for seed in range(n_seeds):
        p = {**params, 'random_state': SEED+seed, 'n_jobs': -1}
        fold_te = np.zeros(len(X_te))
        for tr_idx, val_idx in splits:
            m = ExtraTreesRegressor(**p)
            m.fit(X_tr.iloc[tr_idx], y_[tr_idx])
            oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
            fold_te += m.predict(X_te) / len(splits)
        test_p += fold_te / n_seeds
    return oof, test_p

In [ ]:
print(f'Stacking ({N_FOLDS} fold × {N_SEEDS} seed × 5 model)...\n')

t0 = time.time()
xgb_oof, xgb_test = stack_xgb(xgb_study.best_params, FOLD_SPLITS, X_train, X_test, y)
print(f'  XGB : OOF RMSE = {np.sqrt(mean_squared_error(y, xgb_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
lgb_oof, lgb_test = stack_lgb(lgb_study.best_params, FOLD_SPLITS, X_train, X_test, y)
print(f'  LGB : OOF RMSE = {np.sqrt(mean_squared_error(y, lgb_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
cat_oof, cat_test = stack_cat(cat_study.best_params, FOLD_SPLITS, X_train_cat, X_test_cat, y, cat_features_idx)
print(f'  CAT : OOF RMSE = {np.sqrt(mean_squared_error(y, cat_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
hgb_oof, hgb_test = stack_hgb(hgb_study.best_params, FOLD_SPLITS, X_train, X_test, y)
print(f'  HGB : OOF RMSE = {np.sqrt(mean_squared_error(y, hgb_oof)):.4f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
et_oof, et_test = stack_et(et_study.best_params, FOLD_SPLITS, X_train, X_test, y)
print(f'  ET  : OOF RMSE = {np.sqrt(mean_squared_error(y, et_oof)):.4f}  ({time.time()-t0:.0f}s)')

oof_matrix  = np.column_stack([xgb_oof, lgb_oof, cat_oof, hgb_oof, et_oof])
test_matrix = np.column_stack([xgb_test, lgb_test, cat_test, hgb_test, et_test])
model_names = ['XGB', 'LGB', 'CAT', 'HGB', 'ET']

## 8. Weight Optimization (Nelder-Mead + Multi-start)

In [ ]:
def weighted_rmse(w, preds, y_true):
    return np.sqrt(mean_squared_error(y_true, preds @ w))

n_models = oof_matrix.shape[1]
constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(0.0, 1.0)] * n_models

best_result, best_loss = None, float('inf')
for trial in range(50):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix, y),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_loss:
        best_loss = res.fun; best_result = res

opt_weights = best_result.x
print('Optimum ağırlıklar:')
for n, w in zip(model_names, opt_weights):
    print(f'  {n}: {w:.4f}')
print(f'\nEnsemble OOF RMSE: {best_loss:.4f}')
print(f'En iyi tek model    : {min(np.sqrt(mean_squared_error(y, oof_matrix[:,i])) for i in range(n_models)):.4f}')

ensemble_oof  = oof_matrix @ opt_weights
ensemble_test = test_matrix @ opt_weights

## 9. 🔄 Pseudo-Labeling (v3 yeni)

Stratejik mantık:
1. Ensemble ile test setine tahmin yap
2. Model **birbirine en yakın** olduğu (en düşük std) tahminleri **güvenilir** kabul et
3. Bu örnekleri tahminleriyle birlikte train'e ekle
4. Yeniden eğit → test setine yeniden tahmin yap

Bu, etiketlenmiş veri kümesini etkin olarak büyütür.

In [ ]:
# Model tahminleri arası std → confidence ölçütü
test_stds = test_matrix.std(axis=1)
print(f'Test prediction std: μ={test_stds.mean():.3f}, median={np.median(test_stds):.3f}')

# En güvenilir %30 (std düşük olanlar)
PSEUDO_FRACTION = 0.30
n_pseudo = int(len(test) * PSEUDO_FRACTION)
confident_idx = np.argsort(test_stds)[:n_pseudo]
pseudo_labels = np.clip(ensemble_test[confident_idx], 0, 100)

print(f'\nPseudo-label seçilen: {n_pseudo} örnek (%{PSEUDO_FRACTION*100:.0f})')
print(f'Pseudo-label dağılımı: μ={pseudo_labels.mean():.2f}, σ={pseudo_labels.std():.2f}')

# Genişletilmiş train seti
X_train_aug     = pd.concat([X_train, X_test.iloc[confident_idx]], axis=0).reset_index(drop=True)
X_train_cat_aug = pd.concat([X_train_cat, X_test_cat.iloc[confident_idx]], axis=0).reset_index(drop=True)
y_aug           = np.concatenate([y, pseudo_labels])

# Augmented set için yeni stratified folds
y_aug_bin = pd.qcut(y_aug, q=10, labels=False, duplicates='drop')
skf_aug   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS_AUG = list(skf_aug.split(X_train_aug, y_aug_bin))

print(f'Augmented train shape: {X_train_aug.shape}')
print(f'(Orijinal: {X_train.shape[0]} → Augmented: {X_train_aug.shape[0]})')

In [ ]:
# ── Pseudo-labeled ile yeniden stacking ────────────────────────────────────────
print(f'Pseudo-augmented stacking başlıyor...\n')

t0 = time.time()
xgb_oof2, xgb_test2 = stack_xgb(xgb_study.best_params, FOLD_SPLITS_AUG, X_train_aug, X_test, y_aug)
lgb_oof2, lgb_test2 = stack_lgb(lgb_study.best_params, FOLD_SPLITS_AUG, X_train_aug, X_test, y_aug)
cat_oof2, cat_test2 = stack_cat(cat_study.best_params, FOLD_SPLITS_AUG, X_train_cat_aug, X_test_cat, y_aug, cat_features_idx)
hgb_oof2, hgb_test2 = stack_hgb(hgb_study.best_params, FOLD_SPLITS_AUG, X_train_aug, X_test, y_aug)
et_oof2,  et_test2  = stack_et(et_study.best_params, FOLD_SPLITS_AUG, X_train_aug, X_test, y_aug)
print(f'Augmented stacking süresi: {time.time()-t0:.0f}s')

# Sadece orijinal train satırlarını kullanarak OOF RMSE değerlendir
n_orig = len(y)
xgb_oof2_orig = xgb_oof2[:n_orig]
lgb_oof2_orig = lgb_oof2[:n_orig]
cat_oof2_orig = cat_oof2[:n_orig]
hgb_oof2_orig = hgb_oof2[:n_orig]
et_oof2_orig  = et_oof2[:n_orig]

print(f'\n--- Pseudo-augmented OOF RMSE (sadece original train kısmı) ---')
print(f'  XGB : {np.sqrt(mean_squared_error(y, xgb_oof2_orig)):.4f}')
print(f'  LGB : {np.sqrt(mean_squared_error(y, lgb_oof2_orig)):.4f}')
print(f'  CAT : {np.sqrt(mean_squared_error(y, cat_oof2_orig)):.4f}')
print(f'  HGB : {np.sqrt(mean_squared_error(y, hgb_oof2_orig)):.4f}')
print(f'  ET  : {np.sqrt(mean_squared_error(y, et_oof2_orig)):.4f}')

oof_matrix2  = np.column_stack([xgb_oof2_orig, lgb_oof2_orig, cat_oof2_orig, hgb_oof2_orig, et_oof2_orig])
test_matrix2 = np.column_stack([xgb_test2, lgb_test2, cat_test2, hgb_test2, et_test2])

In [ ]:
# Augmented ensemble weight optimization
best_loss2 = float('inf')
best_w2 = None
for trial in range(50):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix2, y),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_loss2:
        best_loss2 = res.fun; best_w2 = res.x

print(f'Pseudo-augmented ensemble OOF RMSE: {best_loss2:.4f}')
print(f'(Önceki: {best_loss:.4f}, fark: {best_loss-best_loss2:+.4f})')

# Pseudo-labeling iyileşirse onu, yoksa orijinali kullan
if best_loss2 < best_loss:
    final_oof  = oof_matrix2 @ best_w2
    final_test = test_matrix2 @ best_w2
    final_loss = best_loss2
    used_weights = best_w2
    print('\n✓ Pseudo-labeling kullanıldı (RMSE iyileşti)')
else:
    final_oof  = oof_matrix @ opt_weights
    final_test = test_matrix @ opt_weights
    final_loss = best_loss
    used_weights = opt_weights
    print('\n✗ Pseudo-labeling iyileştirmedi → orijinal kullanılıyor')

# Ek olarak: iki versiyonun blend'i
blend = 0.5 * (oof_matrix @ opt_weights) + 0.5 * (oof_matrix2 @ best_w2)
blend_rmse = np.sqrt(mean_squared_error(y, blend))
print(f'\n50/50 Blend RMSE: {blend_rmse:.4f}')
if blend_rmse < final_loss:
    final_oof  = blend
    final_test = 0.5*(test_matrix @ opt_weights) + 0.5*(test_matrix2 @ best_w2)
    final_loss = blend_rmse
    print('✓ 50/50 Blend en iyi sonuç')

final_test = np.clip(final_test, 0, 100)

## 10. Görselleştirme & Submission

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Model RMSE karşılaştırması
rmses = [np.sqrt(mean_squared_error(y, oof_matrix[:,i])) for i in range(n_models)] + [final_loss]
names = model_names + ['Final Ensemble']
colors = ['#5470c6','#91cc75','#fac858','#ee6666','#73c0de','#9a60b4']
axes[0,0].bar(names, rmses, color=colors)
for i, v in enumerate(rmses):
    axes[0,0].text(i, v+0.02, f'{v:.4f}', ha='center', fontsize=9)
axes[0,0].set_title('Model OOF RMSE'); axes[0,0].set_ylabel('RMSE')

axes[0,1].hist(y, bins=50, alpha=0.6, color='steelblue', label='Train (gerçek)', edgecolor='white')
axes[0,1].hist(final_test, bins=50, alpha=0.6, color='coral', label='Test (tahmin)', edgecolor='white')
axes[0,1].set_title('Dağılım'); axes[0,1].legend()

axes[1,0].scatter(y, final_oof, alpha=0.15, s=5, color='steelblue')
axes[1,0].plot([0,100],[0,100], 'r--', lw=1)
axes[1,0].set_xlabel('Gerçek'); axes[1,0].set_ylabel('Tahmin')
axes[1,0].set_title('Ensemble OOF: Gerçek vs Tahmin')

axes[1,1].pie(used_weights, labels=model_names, autopct='%1.1f%%', colors=colors[:5])
axes[1,1].set_title('Ensemble Ağırlıkları')

plt.tight_layout(); plt.show()

In [ ]:
submission = pd.DataFrame({
    'student_id'         : test_ids,
    'career_success_score': final_test
})
submission.to_csv('submission.csv', index=False)

print('═' * 50)
print(f'  Final OOF RMSE: {final_loss:.4f}')
print(f'  Tahmini Skor  : {100 - final_loss:.2f}')
print('═' * 50)
print(f'\nsubmission.csv kaydedildi ✓ ({len(submission)} satır)')
print(submission.head(10))

---
## 📊 v3 Mimari Özeti

| Bileşen | v2 → v3 |
|---------|---------|
| **Role-skill features** | Yok → **8 yeni feature** (+0.12 RMSE doğrulanmış) |
| **NLP boyut** | 80 SVD → **100 SVD** (60 word + 40 char) |
| **Optuna pruner** | MedianPruner → **SuccessiveHalvingPruner (ASHA)** |
| **Trial sayısı** | 40 → **60 (XGB/LGB), 40 (CAT)** |
| **Base models** | 4 → **5** (+ ExtraTrees) |
| **Pseudo-labeling** | Yok → **%30 confident test sample eklenir** |
| **Ensemble blend** | Tek → **Orijinal + Pseudo + 50/50 Blend** karşılaştırma |

### En Kritik Bulgu
Naive feature engineering (interaksiyonlar, polinomlar, null flag) **trees zaten öğreniyor** — gerçek kazanım **rol-yetenek bilgisi** gibi domain feature'lardan geliyor.

### Sırada Ne Var? (Daha Fazla Boost)
- **Multilingual BERT** embeddings (sentence-transformers)
- **Stacked generalization**: 2. seviye Ridge meta + 1. seviye predictions
- **Optuna ile feature selection**: SHAP-based wrapper
- **CatBoost cyclic + symmetric tree** ile farklı bias
- **NN base model** (TabNet, FT-Transformer)